In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from pathlib import Path
from typing import Any

hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]

/home/erfan/miniconda3/envs/hf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
def parse_json_value(value: Any) -> dict:
    """
    Convert either a JSON string or Python dictionary into a dictionary.
    """
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        parsed = json.loads(value)

        if not isinstance(parsed, dict):
            raise ValueError("Tool-call output must be a JSON object.")

        return parsed

    raise TypeError(
        f"Expected output to be dict or JSON string, got {type(value).__name__}"
    )


In [9]:
class QueryGenerator:
    def __init__(self, model, tokenizer, system_prompt, max_new_tokens=254, temperature=0.7):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def build_text(self, user_prompt):
        messages = self.build_messages(user_prompt)

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True,
        )

    def generate(self, user_prompt):
        text = self.build_text(user_prompt)

        model_inputs = self.tokenizer(
            [text],
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p= 0.8,
            top_k=20,
            min_p=0 
            
        )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        model_output = self.tokenizer.decode(
            output_ids,
            # skip_special_tokens=True,
        ).strip("\n")

        return model_output



In [3]:
SYSTEM_PROMPT = (
    "Convert the user's stock-price request into one get_prices function call. "
    "Return only the function call with valid arguments. Do not answer the request."
)


In [4]:
model_path = "./models/qwen3_price_merged"
# model_name = "meta-llama/Llama-3.2-3B-Instruct"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    token=hf_token,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto",
    
)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 15089.53it/s]


In [6]:
query_generator= QueryGenerator(model,tokenizer,SYSTEM_PROMPT, max_new_tokens=328,temperature=0.7)


In [11]:
user_prompts =["Please tell me the weekly price of TESLA from June 2020 ti july 2023","What is the capital of france"]



In [12]:
for user_prompt in user_prompts:
    model_output = query_generator.generate(user_prompt)
    print(model_output)
    print("*"*100)

<tool_call>
{"name": "get_prices", "arguments": {"queries": [{"symbols": ["TSLA"], "timeframe": ["weekly"], "start_month": "June", "start_year": 2020, "end_month": "July", "end_year": 2023}]}}
</tool_call>
****************************************************************************************************
<tool_call>
{"name": "get_prices", "arguments": {"queries": [{"symbols": ["FR"], "timeframe": ["daily"], "language": "english"}]}}
</tool_call>
****************************************************************************************************
